# Census: what is actually in this data

Reads the cached files in `data/raw/`, no network calls. Two files per HS code:

| File | One row is | Used for |
|---|---|---|
| `census_hs_annual_{code}.json` | one country, one month, for that code | mode shares by country (air, vessel, container) |
| `census_porths_vessel_{code}.json` | one US port, one month, for that code | which US coast the goods land on |

Kernel should be `.venv`.

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

# Repo root = the first folder, going up from here, that contains ingest/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ingest").is_dir())
RAW = ROOT / "data" / "raw"
print(RAW)


def read_census(name):
    """One cached Census file as a DataFrame. Census returns a header row then
    data rows, all as strings. It echoes query params back as extra columns,
    so I_COMMODITY appears twice; the second copy is renamed *_echo."""
    header, *rows = json.loads((RAW / name).read_text())
    seen, cols = {}, []
    for h in header:
        seen[h] = seen.get(h, 0) + 1
        cols.append(h if seen[h] == 1 else f"{h}_echo")
    df = pd.DataFrame(rows, columns=cols)
    for c in df.columns:
        if c.endswith("_VAL_YR"):
            df[c] = pd.to_numeric(df[c])
    print(f"{name}: {len(df):,} rows")
    return df


CODE = "854231"   # integrated circuits, processors. Change this to look at another code.

/Users/akash/Desktop/import-concentration-risk/data/raw


## 1. Imports by country: the raw rows

Columns:

- `CTY_CODE`, `CTY_NAME`: Census's own 4-digit country code and name.
- `SUMMARY_LVL`: `DET` is a real country row; `CGP` is a country **group** (European Union, Pacific Rim...). Groups double count, so staging keeps `DET` only.
- `GEN_VAL_YR`: general imports, all modes, in dollars.
- `AIR_VAL_YR`: the part that came by air.
- `VES_VAL_YR`: the part that came by ship, any kind of ship.
- `CNT_VAL_YR`: the part of `VES_VAL_YR` that came in containers. It's a subset of vessel, not a separate mode.
- `time`: the month, `YYYY-MM`.
- `COMM_LVL`, `I_COMMODITY_echo`: Census repeating the query back. Ignore.

In [2]:
hs = read_census(f"census_hs_annual_{CODE}.json")
print(hs.dtypes)
hs.sample(5, random_state=1)

census_hs_annual_854231.json: 9,337 rows
I_COMMODITY           str
CTY_CODE              str
CTY_NAME              str
SUMMARY_LVL           str
GEN_VAL_YR          int64
AIR_VAL_YR          int64
VES_VAL_YR          int64
CNT_VAL_YR          int64
COMM_LVL              str
I_COMMODITY_echo      str
time                  str
dtype: object


,I_COMMODITY,CTY_CODE,CTY_NAME,SUMMARY_LVL,GEN_VAL_YR,AIR_VAL_YR,VES_VAL_YR,CNT_VAL_YR,COMM_LVL,I_COMMODITY_echo,time
8674,854231,2250,PANAMA,DET,2637,2637,0,0,HS6,854231,2018-07
45,854231,5200,UNITED ARAB EMIRATES,DET,105749,105749,0,0,HS6,854231,2024-04
685,854231,5570,MALAYSIA,DET,6652527188,6517126754,14467466,14378540,HS6,854231,2025-09
7095,854231,0014,PACIFIC RIM COUNTRIES,CGP,9966341821,9517025276,122621376,121116191,HS6,854231,2023-10
1227,854231,5800,"KOREA, SOUTH",DET,318053943,295111028,661354,179784,HS6,854231,2019-07


In [3]:
print(hs["SUMMARY_LVL"].value_counts(), "\n")
print("Some country groups (CGP rows), which staging drops:")
print(hs.loc[hs["SUMMARY_LVL"] == "CGP", ["CTY_CODE", "CTY_NAME"]].drop_duplicates().head(10).to_string(index=False))

SUMMARY_LVL
DET    7506
CGP    1831
Name: count, dtype: int64 

Some country groups (CGP rows), which staging drops:
CTY_CODE                        CTY_NAME
    0003                  EUROPEAN UNION
    1XXX                   NORTH AMERICA
    4XXX                          EUROPE
    7XXX                          AFRICA
    0014           PACIFIC RIM COUNTRIES
    5XXX                            ASIA
    0017                        CAFTA-DR
    6XXX           AUSTRALIA AND OCEANIA
    0020                   USMCA (NAFTA)
    0021 TWENTY LATIN AMERICAN REPUBLICS


## 2. The year-to-date trap

`_YR` means **year to date**. The March row is January + February + March. So December is the full year, and adding up the months counts January twelve times.

In [4]:
dec_2023 = hs[(hs["time"] == "2023-12") & (hs["SUMMARY_LVL"] == "DET") & (hs["CTY_CODE"] != "-")]
top_country = dec_2023.nlargest(1, "GEN_VAL_YR").iloc[0]
print(f"Largest supplier of {CODE} in 2023: {top_country['CTY_NAME']} ({top_country['CTY_CODE']})\n")

one = hs[
    (hs["CTY_CODE"] == top_country["CTY_CODE"])
    & hs["time"].str.startswith("2023")
    & (hs["SUMMARY_LVL"] == "DET")
].sort_values("time")
print(one[["time", "GEN_VAL_YR", "AIR_VAL_YR", "VES_VAL_YR", "CNT_VAL_YR"]].to_string(index=False))

summed = one["GEN_VAL_YR"].sum()
december = one.loc[one["time"] == "2023-12", "GEN_VAL_YR"].iloc[0]
print(f"\nSum of the 12 rows: ${summed:,.0f}")
print(f"December row:       ${december:,.0f}   <- the real annual total")
print(f"Summing inflates by {summed / december:.1f}x")

Largest supplier of 854231 in 2023: MALAYSIA (5570)

   time  GEN_VAL_YR  AIR_VAL_YR  VES_VAL_YR  CNT_VAL_YR
2023-01   480962793   426311916      397293      397293
2023-02   989935148   928725699      711697      711697
2023-03  1587160943  1489245284     2070326     2070326
2023-04  1986924058  1877956248     3204515     3204515
2023-05  2447325130  2317725280    10826361    10826361
2023-06  2975237291  2803910784    12498288    12498288
2023-07  3605861016  3409457795    12750921    12750921
2023-08  4043932723  3800531257    13904623    13904623
2023-09  4688465851  4431911593    15071606    15071606
2023-10  5226625214  4956750543    16543441    16543441
2023-11  5726847260  5421750338    18011623    18011623
2023-12  6631297499  6307251714    18923879    18923879

Sum of the 12 rows: $40,390,574,926
December row:       $6,631,297,499   <- the real annual total
Summing inflates by 6.1x


## 3. The '-' row

`CTY_CODE` = `-` is Census's grand total across all countries, and it's tagged `DET` like a real country. Keep it in and every total doubles. Staging drops it, but it's useful as a check: it should equal the sum of the country rows.

In [5]:
dash = hs[(hs["time"] == "2023-12") & (hs["CTY_CODE"] == "-")]
print(dash[["CTY_CODE", "CTY_NAME", "SUMMARY_LVL", "GEN_VAL_YR"]].to_string(index=False))
print(f"\nSum of DET country rows, Dec 2023: ${dec_2023['GEN_VAL_YR'].sum():,.0f}")

CTY_CODE                CTY_NAME SUMMARY_LVL  GEN_VAL_YR
       - TOTAL FOR ALL COUNTRIES         DET 20140980381

Sum of DET country rows, Dec 2023: $20,140,980,381


## 4. December 2023, top 10 countries, how the goods arrived

This is the shape the exposure calculation uses: each country's own container value.

In [7]:
top = dec_2023.nlargest(10, "GEN_VAL_YR")[["CTY_CODE", "CTY_NAME", "GEN_VAL_YR", "AIR_VAL_YR", "VES_VAL_YR", "CNT_VAL_YR"]].copy()
for mode in ["AIR", "VES", "CNT"]:
    top[f"{mode.lower()}_%"] = (top[f"{mode}_VAL_YR"] / top["GEN_VAL_YR"] * 100).round(1)
top["land_other_%"] = (100 - top["air_%"] - top["ves_%"]).round(1)
top

,CTY_CODE,CTY_NAME,GEN_VAL_YR,AIR_VAL_YR,VES_VAL_YR,CNT_VAL_YR,air_%,ves_%,cnt_%,land_other_%
664,5570,MALAYSIA,6631297499,6307251714,18923879,18923879,95.1,0.3,0.3,4.6
1472,5830,TAIWAN,2759515659,2696522264,15999555,15659413,97.7,0.6,0.6,1.7
6406,5081,ISRAEL,1958686794,1957800661,0,0,100.0,0.0,0.0,0.0
3031,4190,IRELAND,1668570678,1668505651,65027,41185,100.0,0.0,0.0,0.0
509,5520,VIETNAM,1653321643,1636914865,974344,974344,99.0,0.1,0.1,0.9
8647,2230,COSTA RICA,1091264182,1056123153,175298,111875,96.8,0.0,0.0,3.2
1178,5700,CHINA,969770644,936916087,14502637,13571800,96.6,1.5,1.4,1.9
1280,5800,"KOREA, SOUTH",874969287,856869697,8669826,8513423,97.9,1.0,1.0,1.1
1568,5880,JAPAN,513237083,421761596,85768235,85691808,82.2,16.7,16.7,1.1
413,5490,THAILAND,480345062,447001748,12544560,12533030,93.1,2.6,2.6,4.3


## 5. Imports by US port (vessel file)

Same code, but one row per **US port of entry** instead of per country. There is no country column in this file, which is why coast share is per code, not per country.

- `PORT`, `PORT_NAME`: Census 4-digit port code. The first two digits are the customs district.
- `GEN_VAL_YR`: everything entering at that port, all modes (a land border crossing has lots of this).
- `VES_VAL_YR`: the part that arrived by ship. This is what coast shares use.

In [8]:
porths = read_census(f"census_porths_vessel_{CODE}.json")
porths.sample(5, random_state=1)

census_porths_vessel_854231.json: 9,836 rows


,I_COMMODITY,PORT,PORT_NAME,SUMMARY_LVL,GEN_VAL_YR,VES_VAL_YR,COMM_LVL,I_COMMODITY_echo,time
260,854231,1012,"JFK INTERNATIONAL AIRPORT, NY",DET,39066784,0,HS6,854231,2023-04
2754,854231,2383,"VALLEY INTERNATIONAL AIRPORT, HARLINGEN, TX",DET,255839,0,HS6,854231,2024-08
6663,854231,4102,"CINCINNATI-LAWRENCEBURG, OH",DET,20505630,0,HS6,854231,2019-12
473,854231,1108,"PHILADELPHIA INTERNATIONAL AIRPORT, PA",DET,21098045,0,HS6,854231,2018-11
9278,854231,3801,"DETROIT, MI",DET,2734323,0,HS6,854231,2024-11


In [9]:
dec_ports = porths[(porths["time"] == "2023-12") & (porths["SUMMARY_LVL"] == "DET") & (porths["PORT"] != "-")].copy()
dec_ports["share_of_vessel_%"] = (dec_ports["VES_VAL_YR"] / dec_ports["VES_VAL_YR"].sum() * 100).round(2)
dec_ports.nlargest(10, "VES_VAL_YR")[["PORT", "PORT_NAME", "GEN_VAL_YR", "VES_VAL_YR", "share_of_vessel_%"]]

,PORT,PORT_NAME,GEN_VAL_YR,VES_VAL_YR,share_of_vessel_%
3731,2704,"LOS ANGELES, CA",107398043,104252726,55.45
795,1401,"NORFOLK-NEWPORT NEWS, VA",21806592,21806592,11.60
4990,3002,"TACOMA, WA",15658452,15658452,8.33
172,1003,"NEWARK, NJ",56194796,12342410,6.56
1362,1703,"SAVANNAH, GA",11839164,11567333,6.15
1154,1601,"CHARLESTON, SC",7710316,7710316,4.10
3827,2709,"LONG BEACH, CA",4497605,4386937,2.33
4894,3001,"SEATTLE, WA",2941018,2676483,1.42
4422,2811,"OAKLAND, CA",2236384,2225740,1.18
7743,5301,"HOUSTON, TX",15537574,2153006,1.15


## 6. Contrast: a code that ships

Integrated circuits fly. Solar modules (854143, only exists from 2022) sail. Same table as section 4, different code.

In [10]:
solar = read_census("census_hs_annual_854143.json")
s = solar[(solar["time"] == "2023-12") & (solar["SUMMARY_LVL"] == "DET") & (solar["CTY_CODE"] != "-")]
s = s.nlargest(10, "GEN_VAL_YR")[["CTY_NAME", "GEN_VAL_YR", "AIR_VAL_YR", "CNT_VAL_YR"]].copy()
s["air_%"] = (s["AIR_VAL_YR"] / s["GEN_VAL_YR"] * 100).round(1)
s["cnt_%"] = (s["CNT_VAL_YR"] / s["GEN_VAL_YR"] * 100).round(1)
s

census_hs_annual_854143.json: 2,739 rows


,CTY_NAME,GEN_VAL_YR,AIR_VAL_YR,CNT_VAL_YR,air_%,cnt_%
210,VIETNAM,4884804221,2436949,4745033448,0.0,97.1
162,THAILAND,4110293941,491286,3935805584,0.0,95.8
324,MALAYSIA,2685320307,308153,2612942027,0.0,97.3
276,CAMBODIA,2367980081,258067,2359384154,0.0,99.6
49,INDIA,1817194780,1211252,1812826923,0.1,99.8
564,"KOREA, SOUTH",1148170499,407600,1147562940,0.0,99.9
2574,MEXICO,441296000,0,285010,0.0,0.1
372,SINGAPORE,341787500,17251,341096538,0.0,99.8
420,INDONESIA,182450577,453539,181997038,0.2,99.8
1623,TURKEY,167703376,13375,167690001,0.0,100.0


## What I noticed

- 